# Silver

Este notebook corresponde à **camada Silver** do pipeline de dados, sendo responsável pela padronização inicial e preparação dos dados provenientes da camada Bronze, disponíveis na tabela **`workspace.bronze.tb_Sinistros_Transito_open_data`** no Databricks.

Nesta etapa, os dados são carregados e passam por um processo de **inferência e ajuste de tipos**, utilizando o **Pandas**. A conversão é realizada com foco em transformar .... TERMINAR

Após essa etapa, os dados são convertidos novamente para um DataFrame Spark e persistidos no Data Lake no formato **Delta Lake (baseado em Parquet)**, garantindo eficiência de leitura, compressão e escalabilidade.



In [0]:
from pyspark.sql import functions as F
import pandas as pd

## Criação do Pandas Dataframe

In [0]:
df_pandas = spark.sql("""
    SELECT * 
    FROM workspace.bronze.tb_Sinistros_Transito_open_data
""").toPandas()

In [0]:
df_pandas.head()


In [0]:
df_pandas.info()

## Vamos verificar se possui linhas duplicadas

In [0]:
df_pandas.duplicated()

## Remoção de duplicatas 

### No nosso caso, não tem duplicatas. Mas caso houvesse, poderíamos usar o comando abaixo:

In [0]:
df_pandas = df_pandas.drop_duplicates()

### Vamos remover as colunas vazias 

In [0]:
# vamos remover algumas colunas
df_pandas.drop(['acidente_verificado', 'tempo_clima', 'situacao_semaforo', 'sinalizacao', 'condicao_via', 'conservacao_via', 'ponto_controle', 'situacao_placa', 'velocidade_max_via', 'mao_direcao', 'divisao_via1', 'divisao_via2', 'num_semaforo', 'sentido_via'], axis=1, inplace=True)

In [0]:
df_pandas.display()

### A coluna com a informação de protocolo também será removida, pois não traz nenhuma informação útil para futuras análises. 

In [0]:
df_pandas.drop(['Protocolo'], axis=1, inplace=True)

# Algumas colunas, faltam mais  de 50% das informações. Vamos removê-las. 

In [0]:
df_pandas.drop(['detalhe_endereco_acidente', 'numero'], axis=1, inplace=True)

In [0]:
df_pandas.display()

### Algumas colunas deveriam ser numéricas, mas estão como string. Vamos fazer a transformação para inteiros: 1,0 -> 1, 2,0 -> 2, ... 

In [0]:
colunas_numericas = [
    "auto", "moto", "ciclom", "ciclista", "pedestre",
    "onibus", "caminhao", "viatura", "outros",
    "vitimas", "vitimasfatais"]

for col in colunas_numericas:
    df_pandas[col] = (
        df_pandas[col]
        .astype(str)
        .str.replace(",", ".")
        .pipe(pd.to_numeric, errors="coerce")
        .fillna(0)
        .astype(int))

In [0]:
df_pandas.display()

### Vamos pradronizar valores nulos

In [0]:
# Padroniza valores nulos na coluna 'complemento' para "NA"
# Isso evita problemas em análises e garante consistência textual
df_pandas["complemento"] = df_pandas["complemento"].fillna("NA")



In [0]:
# Vamos padronizar outras colunas texto

colunas_texto = ["bairro", "endereco", "bairro_cruzamento", "tipo"]

df_pandas[colunas_texto] = df_pandas[colunas_texto].fillna("NA")

In [0]:
df_pandas.display()

In [0]:
df_pandas.info()

### Antes, 1.6 MB. Agora, ~934 KB. Isto representa uma redução de mais de 40% no tamanho dos dados. 

### Persistência dos dados na camada Silver

#### Gravação dos dados em formato Delta Lake (Parquet)

Nesta etapa, os dados tratados e transformados na camada Silver são persistidos no Data Lake utilizando o formato **Delta Lake**, que é construído sobre arquivos **Parquet**.

O Parquet é um formato de armazenamento colunar que proporciona **alta compressão** e **leitura eficiente**, permitindo que apenas as colunas necessárias sejam processadas. Isso resulta em melhor desempenho e redução de custo computacional, especialmente em grandes volumes de dados.

O Delta Lake complementa o Parquet ao adicionar funcionalidades essenciais para pipelines de dados modernos, como:
- controle de versão (time travel)  
- transações ACID (maior confiabilidade)  
- suporte a evolução de schema  
- base para cargas incrementais futuras  

Embora o cenário atual não envolva ingestão incremental, a utilização do Delta Lake garante que o pipeline esteja preparado para evoluções futuras, mantendo escalabilidade e robustez.

Após a gravação dos dados, será criada uma **tabela no catálogo do Databricks (Unity Catalog)**, permitindo consultas via SQL diretamente sobre os dados armazenados no Data Lake, sem necessidade de duplicação.

Essa abordagem assegura um fluxo de dados eficiente, confiável e alinhado com boas práticas de engenharia de dados.

In [0]:
%sql
SHOW VOLUMES IN workspace.silver;

### Vamos criar o volume na camada Silver

Quando rodei a primeira vez, tive que criar o volume. 

In [0]:
%sql
CREATE VOLUME workspace.silver.dbs;

In [0]:
# Convertendo o DataFrame do Pandas para DataFrame do Spark
df_spark = spark.createDataFrame(df_pandas)

# Grava os dados no formato Delta Lake na camada Silver (Data Lake),
# sobrescrevendo os dados e permitindo evolução do schema
df_spark.write.format("delta") \
  .mode("overwrite") \
  .option("mergeSchema","true") \
  .save("/Volumes/workspace/silver/dbs/Sinistros_Transito/Sinistros_Transito_open_data")

In [0]:
# Vamos visualizar e validar dados ingeridos na camada Silver

spark.read.format("delta").load(
    "/Volumes/workspace/silver/dbs/Sinistros_Transito/Sinistros_Transito_open_data"
).createOrReplaceTempView("tb_Sinistros_Transito_open_data")

spark.sql("""SELECT * 
          FROM tb_Sinistros_Transito_open_data 
          LIMIT 450;"""
          ).display()